In [1]:
#子任务 1：BERT + LoRA 微调
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from datasets import load_dataset
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model
import evaluate
import numpy as np


#硬件与基础配置 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")


#加载数据集
dataset = load_dataset("imdb")  # 自动下载IMDb影评数据集

#数据预处理
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_datasets = dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#加载模型 + 配置LoRA
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    #torch_dtype=torch.float16  # 混合精度，省显存
).to(device)

#配置LoRA
lora_config = LoraConfig(
    task_type="SEQ_CLS",  # 序列分类任务
    r=8,
    lora_alpha=16,
    target_modules=["query", "key", "value"],
    lora_dropout=0.1,
)

#包装模型，仅LoRA参数可训练
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # 输出可训练参数量占比


#评估函数
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

#训练参数（针对4060 8G优化）
training_args = TrainingArguments(
    output_dir="./bert-lora-imdb",
    learning_rate=2e-4,  # LoRA学习率可设大一点
    per_device_train_batch_size=4,  # 批次大小4，梯度累积补
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # 梯度累积2步，等效batch=8
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    #fp16=True,  # 强制混合精度，省显存关键
    logging_steps=10,
)

2026-03-28 19:12:19.597532: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-28 19:12:19.597712: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-28 19:12:19.642251: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


使用设备: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 443,906 || all params: 109,927,684 || trainable%: 0.4038


In [2]:
#开始训练
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(1000)),
    eval_dataset=tokenized_datasets["test"].shuffle(seed=42).select(range(1000)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.538700,0.508865,0.746000
2,0.287000,0.389465,0.836000


TrainOutput(global_step=250, training_loss=0.5334090232849121, metrics={'train_runtime': 133.4801, 'train_samples_per_second': 14.984, 'train_steps_per_second': 1.873, 'total_flos': 436007262063552.0, 'train_loss': 0.5334090232849121, 'epoch': 2.0})

In [3]:
#简单测试
def classify_review(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    return "正面" if prediction == 1 else "负面"

print(classify_review("This movie is absolutely fantastic!"))  # 测试
print(classify_review("This movie is so bad!"))
print(classify_review("This movie is good!"))
print(classify_review("I don't like it!"))
print(classify_review("My friend like it"))

print(classify_review("This movie completely blew me away from start to finish. The storytelling is incredibly heartfelt, with well-developed characters that feel real and relatable. Every performance feels genuine and full of emotion, especially the lead actor who portrays vulnerability and strength perfectly. The pacing is smooth, the cinematography is beautiful, and the plot delivers meaningful messages without being forced. By the end, I was genuinely moved and left with a warm, satisfying feeling—easily one of the best films I’ve watched in a long time."))
print(classify_review("I found this movie extremely disappointing and a total waste of time. The plot is messy and full of plot holes, with no clear direction or logical flow. The characters are one-dimensional and make ridiculous decisions that no real person would ever choose. The acting feels stiff and unconvincing, and the dialogue is cringeworthy throughout. Even the visuals are dull and uninspired. I kept waiting for it to get better, but it only became more tedious—definitely not worth watching at all."))

正面
正面
正面
正面
正面
正面
负面


In [4]:
#数据预处理（适配PT）
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=492)
tokenized_datasets = dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [5]:
#子任务 2：BERT + Prompt Tuning 微调
from peft import PromptTuningConfig, PromptTuningInit

model_prompt = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(device)

# 配置Prompt Tuning
prompt_config = PromptTuningConfig(
    task_type="SEQ_CLS",
    prompt_tuning_init=PromptTuningInit.TEXT,  # 用文本初始化软提示
    num_virtual_tokens=20,  # 软提示长度
    #prompt_tuning_init_text="Classify if the review is positive or negative: [SEP]",
    prompt_tuning_init_text="Is this movie review positive or negative? ",
    tokenizer_name_or_path="bert-base-uncased",
)

# 包装模型
model = get_peft_model(model_prompt, prompt_config)
model.print_trainable_parameters()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(1000)),
    eval_dataset=tokenized_datasets["test"].shuffle(seed=42).select(range(1000)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 16,898 || all params: 109,500,676 || trainable%: 0.0154


Epoch,Training Loss,Validation Loss,Accuracy
1,0.716300,0.676098,0.593000
2,0.685500,0.670762,0.611000


TrainOutput(global_step=250, training_loss=0.6857835941314697, metrics={'train_runtime': 110.2228, 'train_samples_per_second': 18.145, 'train_steps_per_second': 2.268, 'total_flos': 424288584549024.0, 'train_loss': 0.6857835941314697, 'epoch': 2.0})

In [6]:
print(classify_review("This movie is absolutely fantastic!"))  # 测试
print(classify_review("This movie is so bad!"))
print(classify_review("This movie is good!"))
print(classify_review("I don't like it!"))
print(classify_review("My friend like it"))

负面
负面
负面
负面
负面
